# Big Data Lab 02 (CO3137) - Spark Structured Streaming

Notebook này tổng hợp lại 4 bài tập (Exercise 1 -> Exercise 4) để tiện chạy thử và demo từng ô trên JupyterLab.
Trước khi chạy, đảm bảo bạn đã chạy `python source/setup_kafka.py` để đẩy dữ liệu vào 3 topic `Lab1_movies`, `Lab1_ratings`, `Lab1_tags` (hoặc sửa `KAFKA_SERVERS` trong `source/config.py` khi kết nối Kafka phòng Lab).

## Exercise 1: Chuẩn bị dữ liệu phim (Bắt buộc)
- Khởi tạo `SparkSession`
- Đọc 3 topic Kafka và ép kiểu đúng schema cho `movies`, `ratings`, `tags`

In [1]:
import time
from pyspark.sql.functions import (
    avg, col, count, dense_rank, explode, round, split, window
)
from pyspark.sql.window import Window

from ex1_prepare_data import (
    create_spark,
    read_movies,
    read_ratings_stream,
    read_tags_stream,
)

spark = create_spark("Lab02-Notebook")

movies_df = read_movies(spark).cache()
ratings_df = read_ratings_stream(spark)
tags_df = read_tags_stream(spark)

print("=== Schema movies (static) ===")
movies_df.printSchema()
movies_df.show(5, truncate=False)

print("=== Schema ratings (stream) ===")
ratings_df.printSchema()

print("=== Schema tags (stream) ===")
tags_df.printSchema()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /tmp/.ivy2/cache
The jars for the packages stored in: /tmp/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d051c24e-f6a2-4f6c-9b10-2944acafa51a;1.0
	confs: [default]


	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.1 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found org.slf4j#slf4j-api;2.0.16 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.1 in central
	found org.apache.hadoop#hadoop-client-api;3.4.1 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central


	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 in central
	found org.apache.commons#commons-pool2;2.12.0 in central
:: resolution report :: resolve 382ms :: artifacts dl 14ms
	:: modules in use:
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	org.apache.commons#commons-pool2;2.12.0 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.4.1 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.4.1 from central in [default]
	org.apache.kafka#kafka-clients;3.9.1 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.1 from central in [default]
	org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.1 from central in [default]
	org.lz4#lz4-java;1.8.0 from central in [default]
	org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 from central in [default]
	org.slf4j#slf4j-api;2.0.16 from central in [default]
	org.xerial.snappy#snappy-java;1.1.10.7 from central in [default]
	------------------------

26/11/18 09:51:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


=== Schema movies (static) ===
root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)



+-------+----------------------------------+----------------------------+
|movieId|title                             |genres                      |
+-------+----------------------------------+----------------------------+
|12     |Dracula: Dead and Loving It (1995)|Comedy|Horror               |
|13     |Balto (1995)                      |Adventure|Animation|Children|
|14     |Nixon (1995)                      |Drama                       |
|18     |Four Rooms (1995)                 |Comedy                      |
|25     |Leaving Las Vegas (1995)          |Drama|Romance               |
+-------+----------------------------------+----------------------------+
only showing top 5 rows
=== Schema ratings (stream) ===
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)

=== Schema tags (stream) ===
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true

## Exercise 2: Which genres are hot right now?
- Static-stream join giữa `ratings` (stream) và `movies` (static)
- Tách cột `genres` theo dấu `|`, gom nhóm theo từng thể loại
- Cập nhật ra console mỗi 5 giây (chạy thử 20 giây rồi dừng để sang ô tiếp theo)

In [2]:
joined_ex2 = ratings_df.join(
    movies_df.select("movieId", "genres"),
    on="movieId",
    how="inner",
)

hot_genres_df = (
    joined_ex2
    .withColumn("genre", explode(split(col("genres"), "\\|")))
    .filter((col("genre") != "") & (col("genre") != "(no genres listed)"))
    .groupBy("genre")
    .agg(
        count("*").alias("rating_count"),
        round(avg("rating"), 2).alias("avg_rating"),
    )
)

def show_hot_genres(batch_df, batch_id):
    print(f"Batch: {batch_id}")
    batch_df.orderBy(col("rating_count").desc()).show(truncate=False)

q2 = (
    hot_genres_df.writeStream
    .outputMode("complete")
    .foreachBatch(show_hot_genres)
    .trigger(processingTime="5 seconds")
    .start()
)
# Chạy thử 20 giây rồi dừng (bỏ số 20 và dòng q2.stop() nếu muốn chạy liên tục)
q2.awaitTermination(20)
q2.stop()

26/11/18 09:51:17 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-188c429c-e8bd-442d-9e05-522d172e81e7. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/11/18 09:51:17 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Batch: 0


+---------+------------+----------+
|genre    |rating_count|avg_rating|
+---------+------------+----------+
|Action   |37          |4.19      |
|Comedy   |36          |4.22      |
|Adventure|33          |4.36      |
|Drama    |28          |4.39      |
|Thriller |23          |4.09      |
|Crime    |18          |4.11      |
|Romance  |18          |4.33      |
|Children |17          |4.59      |
|Fantasy  |17          |4.53      |
|Musical  |15          |4.87      |
|Sci-Fi   |14          |4.21      |
|War      |12          |4.33      |
|Animation|11          |4.55      |
|Horror   |6           |3.33      |
|Mystery  |6           |4.67      |
|Western  |4           |4.25      |
+---------+------------+----------+



Batch: 1


+---------+------------+----------+
|genre    |rating_count|avg_rating|
+---------+------------+----------+
|Action   |77          |4.3       |
|Adventure|73          |4.38      |
|Comedy   |69          |4.26      |
|Drama    |57          |4.54      |
|Thriller |47          |4.15      |
|Crime    |41          |4.37      |
|Fantasy  |41          |4.32      |
|Children |39          |4.59      |
|Sci-Fi   |35          |4.17      |
|Animation|28          |4.71      |
|Romance  |25          |4.32      |
|Musical  |22          |4.68      |
|War      |17          |4.41      |
|Horror   |16          |3.38      |
|Mystery  |15          |4.27      |
|Western  |6           |4.17      |
|Film-Noir|1           |5.0       |
+---------+------------+----------+



Batch: 2
+-----------+------------+----------+
|genre      |rating_count|avg_rating|
+-----------+------------+----------+
|Action     |115         |4.2       |
|Drama      |101         |3.82      |
|Comedy     |99          |3.96      |
|Adventure  |99          |4.2       |
|Thriller   |72          |4.08      |
|Sci-Fi     |59          |4.19      |
|Crime      |57          |4.12      |
|Fantasy    |51          |4.23      |
|Children   |47          |4.12      |
|Animation  |33          |4.18      |
|Romance    |32          |3.72      |
|War        |28          |3.79      |
|Horror     |26          |3.83      |
|Musical    |23          |4.5       |
|Mystery    |21          |4.19      |
|Western    |8           |4.19      |
|IMAX       |4           |3.75      |
|Documentary|3           |4.33      |
|Film-Noir  |1           |5.0       |
+-----------+------------+----------+



Batch: 3
+-----------+------------+----------+
|genre      |rating_count|avg_rating|
+-----------+------------+----------+
|Drama      |147         |3.76      |
|Comedy     |147         |3.86      |
|Action     |126         |4.15      |
|Adventure  |115         |4.17      |
|Thriller   |88          |4.0       |
|Crime      |71          |4.07      |
|Sci-Fi     |65          |4.12      |
|Romance    |64          |3.66      |
|Fantasy    |63          |4.12      |
|Children   |54          |4.06      |
|Animation  |37          |4.16      |
|Musical    |32          |4.39      |
|War        |31          |3.77      |
|Mystery    |29          |4.07      |
|Horror     |28          |3.88      |
|Western    |15          |3.83      |
|Documentary|5           |4.2       |
|IMAX       |5           |3.6       |
|Film-Noir  |3           |4.67      |
+-----------+------------+----------+



Batch: 4
+-----------+------------+----------+
|genre      |rating_count|avg_rating|
+-----------+------------+----------+
|Drama      |208         |3.65      |
|Comedy     |198         |3.75      |
|Action     |140         |4.04      |
|Adventure  |127         |4.07      |
|Thriller   |108         |3.92      |
|Romance    |87          |3.53      |
|Crime      |82          |4.02      |
|Sci-Fi     |71          |3.96      |
|Fantasy    |68          |4.08      |
|Children   |56          |4.06      |
|Mystery    |43          |3.86      |
|Animation  |39          |4.15      |
|Musical    |38          |4.38      |
|War        |33          |3.7       |
|Horror     |29          |3.88      |
|Western    |18          |3.97      |
|Film-Noir  |5           |4.2       |
|IMAX       |5           |3.6       |
|Documentary|5           |4.2       |
+-----------+------------+----------+



26/11/18 09:51:37 WARN DAGScheduler: Failed to cancel job group 580597ea-aa23-4bf9-8cd3-8dbef45de2c4. Cannot find active jobs for it.
26/11/18 09:51:37 WARN DAGScheduler: Failed to cancel job group 580597ea-aa23-4bf9-8cd3-8dbef45de2c4. Cannot find active jobs for it.


## Exercise 3: What goes on the Trending Now rail?
- Watermark trên `eventTime` trước khi chia tumbling window 5 phút
- Đếm theo `(window, movieId)`, join lấy `title`
- Dùng `dense_rank()` lấy Top 3 mỗi window (tie-break theo `movieId` tăng dần) qua `foreachBatch` với `complete` mode

In [3]:
ratings_with_time = ratings_df.withColumnRenamed("timestamp", "eventTime")

windowed_counts = (
    ratings_with_time
    .withWatermark("eventTime", "10 minutes")
    .groupBy(window(col("eventTime"), "5 minutes"), col("movieId"))
    .agg(count("*").alias("cnt"))
)

joined_ex3 = windowed_counts.join(
    movies_df.select("movieId", "title"),
    on="movieId",
    how="inner",
)

def process_batch(batch_df, batch_id):
    window_spec = Window.partitionBy("window").orderBy(
        col("cnt").desc(),
        col("movieId").asc(),
    )
    top3_df = (
        batch_df
        .withColumn("rank", dense_rank().over(window_spec))
        .filter(col("rank") <= 3)
        .select("window", "movieId", "title", "cnt", "rank")
        .orderBy(col("window.start").desc(), col("rank").asc())
    )
    print(f"Batch: {batch_id}")
    top3_df.show(30, truncate=False)

q3 = (
    joined_ex3.writeStream
    .outputMode("complete")
    .foreachBatch(process_batch)
    .trigger(processingTime="5 seconds")
    .start()
)
while q3.isActive and q3.lastProgress is None:
    time.sleep(0.5)
q3.explain()
q3.awaitTermination(20)
q3.stop()

26/11/18 09:51:38 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-649dddcd-6593-432e-bd95-6f325708d717. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/11/18 09:51:38 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Batch: 0


+------------------------------------------+-------+-----------------------------------------------------+---+----+
|window                                    |movieId|title                                                |cnt|rank|
+------------------------------------------+-------+-----------------------------------------------------+---+----+
|{2000-07-30 19:05:00, 2000-07-30 19:10:00}|151    |Rob Roy (1995)                                       |1  |1   |
|{2000-07-30 19:05:00, 2000-07-30 19:10:00}|157    |Canadian Bacon (1995)                                |1  |2   |
|{2000-07-30 19:05:00, 2000-07-30 19:10:00}|527    |Schindler's List (1993)                              |1  |3   |
|{2000-07-30 19:00:00, 2000-07-30 19:05:00}|47     |Seven (a.k.a. Se7en) (1995)                          |1  |1   |
|{2000-07-30 19:00:00, 2000-07-30 19:05:00}|163    |Desperado (1995)                                     |1  |2   |
|{2000-07-30 19:00:00, 2000-07-30 19:05:00}|593    |Silence of the Lambs

== Physical Plan ==
*(6) Project [movieId#95, window#836-T600000ms, cnt#830L, title#9]
+- *(6) BroadcastHashJoin [movieId#95], [movieId#11], Inner, BuildRight, false
   :- *(6) HashAggregate(keys=[window#836-T600000ms, movieId#95], functions=[count(1)])
   :  +- StateStoreSave [window#836-T600000ms, movieId#95], state info [ checkpoint = file:/tmp/temporary-649dddcd-6593-432e-bd95-6f325708d717/state, runId = 89ff4951-ac82-403c-a4a9-6824340f9652, opId = 0, ver = 0, numPartitions = 4] stateStoreCkptIds = None, Complete, 0, 0, 2
   :     +- *(4) HashAggregate(keys=[window#836-T600000ms, movieId#95], functions=[merge_count(1)])
   :        +- StateStoreRestore [window#836-T600000ms, movieId#95], state info [ checkpoint = file:/tmp/temporary-649dddcd-6593-432e-bd95-6f325708d717/state, runId = 89ff4951-ac82-403c-a4a9-6824340f9652, opId = 0, ver = 0, numPartitions = 4] stateStoreCkptIds = None, 2
   :           +- *(3) HashAggregate(keys=[window#836-T600000ms, movieId#95], functions=[merge_co

Batch: 1
+------------------------------------------+-------+-----------------------------------------------------+---+----+
|window                                    |movieId|title                                                |cnt|rank|
+------------------------------------------+-------+-----------------------------------------------------+---+----+
|{2000-08-08 07:25:00, 2000-08-08 07:30:00}|2492   |20 Dates (1998)                                      |1  |1   |
|{2000-07-30 19:05:00, 2000-07-30 19:10:00}|151    |Rob Roy (1995)                                       |1  |1   |
|{2000-07-30 19:05:00, 2000-07-30 19:10:00}|157    |Canadian Bacon (1995)                                |1  |2   |
|{2000-07-30 19:05:00, 2000-07-30 19:10:00}|527    |Schindler's List (1993)                              |1  |3   |
|{2000-07-30 19:00:00, 2000-07-30 19:05:00}|47     |Seven (a.k.a. Se7en) (1995)                          |1  |1   |
|{2000-07-30 19:00:00, 2000-07-30 19:05:00}|163    |Desperado (

Batch: 2
+------------------------------------------+-------+------------------------------------------------------+---+----+
|window                                    |movieId|title                                                 |cnt|rank|
+------------------------------------------+-------+------------------------------------------------------+---+----+
|{2015-10-24 19:35:00, 2015-10-24 19:40:00}|80489  |Town, The (2010)                                      |1  |1   |
|{2015-10-24 19:30:00, 2015-10-24 19:35:00}|333    |Tommy Boy (1995)                                      |1  |1   |
|{2015-10-24 19:30:00, 2015-10-24 19:35:00}|1704   |Good Will Hunting (1997)                              |1  |2   |
|{2015-10-24 19:30:00, 2015-10-24 19:35:00}|46970  |Talladega Nights: The Ballad of Ricky Bobby (2006)    |1  |3   |
|{2015-10-24 19:25:00, 2015-10-24 19:30:00}|318    |Shawshank Redemption, The (1994)                      |1  |1   |
|{2015-10-24 19:25:00, 2015-10-24 19:30:00}|3578   |Gla

Batch: 3
+------------------------------------------+-------+------------------------------------------------------+---+----+
|window                                    |movieId|title                                                 |cnt|rank|
+------------------------------------------+-------+------------------------------------------------------+---+----+
|{2015-10-24 19:35:00, 2015-10-24 19:40:00}|80489  |Town, The (2010)                                      |1  |1   |
|{2015-10-24 19:30:00, 2015-10-24 19:35:00}|333    |Tommy Boy (1995)                                      |1  |1   |
|{2015-10-24 19:30:00, 2015-10-24 19:35:00}|1704   |Good Will Hunting (1997)                              |1  |2   |
|{2015-10-24 19:30:00, 2015-10-24 19:35:00}|46970  |Talladega Nights: The Ballad of Ricky Bobby (2006)    |1  |3   |
|{2015-10-24 19:25:00, 2015-10-24 19:30:00}|318    |Shawshank Redemption, The (1994)                      |1  |1   |
|{2015-10-24 19:25:00, 2015-10-24 19:30:00}|3578   |Gla

Batch: 4


+------------------------------------------+-------+------------------------------------------------------+---+----+
|window                                    |movieId|title                                                 |cnt|rank|
+------------------------------------------+-------+------------------------------------------------------+---+----+
|{2015-10-24 19:35:00, 2015-10-24 19:40:00}|80489  |Town, The (2010)                                      |1  |1   |
|{2015-10-24 19:30:00, 2015-10-24 19:35:00}|333    |Tommy Boy (1995)                                      |1  |1   |
|{2015-10-24 19:30:00, 2015-10-24 19:35:00}|1704   |Good Will Hunting (1997)                              |1  |2   |
|{2015-10-24 19:30:00, 2015-10-24 19:35:00}|46970  |Talladega Nights: The Ballad of Ricky Bobby (2006)    |1  |3   |
|{2015-10-24 19:25:00, 2015-10-24 19:30:00}|318    |Shawshank Redemption, The (1994)                      |1  |1   |
|{2015-10-24 19:25:00, 2015-10-24 19:30:00}|3578   |Gladiator (2

26/11/18 09:51:59 WARN DAGScheduler: Failed to cancel job group 89ff4951-ac82-403c-a4a9-6824340f9652. Cannot find active jobs for it.
26/11/18 09:51:59 WARN DAGScheduler: Failed to cancel job group 89ff4951-ac82-403c-a4a9-6824340f9652. Cannot find active jobs for it.


## Exercise 4: Stream-Stream Join: Tags and Ratings
- Đặt watermark trên cả `ratings` và `tags`
- Join 2 luồng theo `movieId` ở chế độ `append`
- Xuất ra `(movieId, tag, rating, ratingTime, tagTime)` mỗi 5 giây

In [4]:
ratings_stream = (
    ratings_df
    .withColumnRenamed("timestamp", "ratingTime")
    .withWatermark("ratingTime", "10 minutes")
    .select("movieId", "rating", "ratingTime")
)

tags_stream = (
    tags_df
    .withColumnRenamed("timestamp", "tagTime")
    .withWatermark("tagTime", "10 minutes")
    .select("movieId", "tag", "tagTime")
)

joined_ex4 = (
    ratings_stream
    .join(tags_stream, on="movieId", how="inner")
    .select("movieId", "tag", "rating", "ratingTime", "tagTime")
)

def show_stream_join(batch_df, batch_id):
    batch_df.persist()
    try:
        row_count = batch_df.count()
        print(f"Batch: {batch_id} (so dong khop: {row_count})")
        batch_df.show(20, truncate=False)
    finally:
        batch_df.unpersist()

q4 = (
    joined_ex4.writeStream
    .outputMode("append")
    .foreachBatch(show_stream_join)
    .trigger(processingTime="5 seconds")
    .start()
)
while q4.isActive and q4.lastProgress is None:
    time.sleep(0.5)
q4.explain()
q4.awaitTermination(20)
q4.stop()

26/11/18 09:51:59 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-3ba9ce9c-625b-4182-985a-c606cb7fe514. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/11/18 09:51:59 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Batch: 0 (so dong khop: 9)
+-------+---------------------------+------+-------------------+-------------------+
|movieId|tag                        |rating|ratingTime         |tagTime            |
+-------+---------------------------+------+-------------------+-------------------+
|110    |beautiful scenery          |4.0   |2000-07-30 18:36:16|2018-06-04 22:49:01|
|110    |epic                       |4.0   |2000-07-30 18:36:16|2018-06-04 22:48:52|
|110    |historical                 |4.0   |2000-07-30 18:36:16|2018-06-04 22:48:43|
|110    |inspirational              |4.0   |2000-07-30 18:36:16|2018-06-04 22:48:47|
|110    |Medieval                   |4.0   |2000-07-30 18:36:16|2018-06-04 22:48:48|
|110    |mel gibson                 |4.0   |2000-07-30 18:36:16|2018-06-04 22:48:41|
|110    |Oscar (Best Cinematography)|4.0   |2000-07-30 18:36:16|2018-06-04 22:48:59|
|110    |revenge                    |4.0   |2000-07-30 18:36:16|2018-06-04 22:48:51|
|110    |sword fight                |4

Batch: 1 (so dong khop: 0)
+-------+---+------+----------+-------+
|movieId|tag|rating|ratingTime|tagTime|
+-------+---+------+----------+-------+
+-------+---+------+----------+-------+

== Physical Plan ==
*(3) Project [movieId#95, tag#115, rating#96, ratingTime#1613-T600000ms, tagTime#1614-T600000ms]
+- StreamingSymmetricHashJoin [movieId#95], [movieId#118], Inner, condition = [ leftOnly = null, rightOnly = null, both = null, full = null ], state info [ checkpoint = file:/tmp/temporary-3ba9ce9c-625b-4182-985a-c606cb7fe514/state, runId = d5d05ebb-ba28-4e94-89c2-c9762cfe207c, opId = 0, ver = 1, numPartitions = 4] stateStoreCkptIds = None, 0, 964983553000, state cleanup [ left = null, right = null ], 2
   :- Exchange hashpartitioning(movieId#95, 4), REQUIRED_BY_STATEFUL_OPERATOR, [plan_id=3317]
   :  +- EventTimeWatermark 16e7c041-ec8c-489e-84c3-a0cde6af429e, ratingTime#1613: timestamp, 10 minutes
   :     +- *(1) Project [cast(cast(data#89.movieId as double) as int) AS movieId#95, cas

Batch: 2 (so dong khop: 40)
+-------+---------------+------+-------------------+-------------------+
|movieId|tag            |rating|ratingTime         |tagTime            |
+-------+---------------+------+-------------------+-------------------+
|60756  |funny          |5.0   |2015-10-24 19:29:40|2015-10-24 19:29:54|
|60756  |Highly quotable|5.0   |2015-10-24 19:29:40|2015-10-24 19:29:56|
|60756  |will ferrell   |5.0   |2015-10-24 19:29:40|2015-10-24 19:29:52|
|89774  |Boxing story   |5.0   |2015-10-24 19:33:09|2015-10-24 19:33:27|
|89774  |MMA            |5.0   |2015-10-24 19:33:09|2015-10-24 19:33:20|
|89774  |Tom Hardy      |5.0   |2015-10-24 19:33:09|2015-10-24 19:33:25|
|60756  |comedy         |5.0   |2015-10-24 19:29:40|2018-06-13 23:59:44|
|60756  |funny          |5.0   |2015-10-24 19:29:40|2018-06-13 23:59:41|
|60756  |will ferrell   |5.0   |2015-10-24 19:29:40|2018-06-13 23:59:39|
|3578   |ancient Rome   |4.0   |2015-10-24 19:28:05|2018-06-04 22:48:24|
|3578   |Epic          

Batch: 3 (so dong khop: 0)
+-------+---+------+----------+-------+
|movieId|tag|rating|ratingTime|tagTime|
+-------+---+------+----------+-------+
+-------+---+------+----------+-------+



Batch: 4 (so dong khop: 2)
+-------+-----------+------+-------------------+-------------------+
|movieId|tag        |rating|ratingTime         |tagTime            |
+-------+-----------+------+-------------------+-------------------+
|2387   |dark comedy|5.0   |2000-07-30 18:52:03|2016-09-20 14:37:05|
|2387   |dark humor |5.0   |2000-07-30 18:52:03|2016-09-20 14:37:03|
+-------+-----------+------+-------------------+-------------------+



Batch: 5 (so dong khop: 3)
+-------+----------------+------+-------------------+-------------------+
|movieId|tag             |rating|ratingTime         |tagTime            |
+-------+----------------+------+-------------------+-------------------+
|260    |oldie but goodie|5.0   |2000-07-30 18:28:00|2018-03-01 10:11:41|
|260    |sci-fi          |5.0   |2000-07-30 18:28:00|2018-03-01 10:11:18|
|260    |Star Wars       |5.0   |2000-07-30 18:28:00|2018-03-01 10:11:48|
+-------+----------------+------+-------------------+-------------------+



26/11/18 09:52:21 WARN DAGScheduler: Failed to cancel job group d5d05ebb-ba28-4e94-89c2-c9762cfe207c. Cannot find active jobs for it.
26/11/18 09:52:21 WARN DAGScheduler: Failed to cancel job group d5d05ebb-ba28-4e94-89c2-c9762cfe207c. Cannot find active jobs for it.
